In [1]:
import os
import pandas as pd
import numpy as np
from scipy.signal import butter, filtfilt

In [2]:
# Butterworth low-pass filter setup
def lowpass_filter(data, cutoff=5, fs=60, order=2):
    nyq = 0.5 * fs  # Nyquist frequency
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)

In [ ]:
def add_orientation_elevation(folder_path):
    print("Working on", folder_path)
    df = pd.read_csv(os.path.join(folder_path, "merged.csv"))
    df = df.loc[~df['walk_mode'].isin(['pavement_down', 'pavement_up'])]  # Filter out specific walk modes

    df = df.dropna()

    # Applying the low-pass filter to relevant columns
    df['acceleration_Pelvis_z_filtered'] = lowpass_filter(df['acceleration_Pelvis_z'])
    df['angularVelocity_Pelvis_x_filtered'] = lowpass_filter(df['angularVelocity_Pelvis_x'])
    df['angularVelocity_Pelvis_y_filtered'] = lowpass_filter(df['angularVelocity_Pelvis_y'])
    df['angularVelocity_Pelvis_z_filtered'] = lowpass_filter(df['angularVelocity_Pelvis_z'])

    # Initialize columns
    df['elevation'] = 0.0
    df['orientation_x'] = 0.0
    df['orientation_y'] = 0.0
    df['orientation_z'] = 0.0

    # Constants
    dt = 1 / 60  # Time interval (60Hz sampling rate)

    # Calculate elevation difference and orientation for each row
    for i in range(1, len(df)):
        # Integrate acceleration to estimate elevation change (simple cumulative integration)
        velocity_z = df['acceleration_Pelvis_z_filtered'].iloc[i-1] * dt
        elevation_diff = velocity_z * dt
        df.loc[i, 'elevation'] = df.loc[i-1, 'elevation'] + elevation_diff

        # Integrate angular velocity to estimate orientation
        df.loc[i, 'orientation_x'] = df.loc[i-1, 'orientation_x'] + df['angularVelocity_Pelvis_x_filtered'].iloc[i-1] * dt
        df.loc[i, 'orientation_y'] = df.loc[i-1, 'orientation_y'] + df['angularVelocity_Pelvis_y_filtered'].iloc[i-1] * dt
        df.loc[i, 'orientation_z'] = df.loc[i-1, 'orientation_z'] + df['angularVelocity_Pelvis_z_filtered'].iloc[i-1] * dt

    # Adding the elevation difference column (difference between current and previous elevation)
    df['elevation_diff'] = df['elevation'].diff().fillna(0)

    df.to_csv(os.path.join(folder_path, "merged.csv"), index=False)

In [ ]:
dataset_path = "data_set"
print("Adding orientation and elevation...")
for course_folder in os.listdir(dataset_path):
    course_folder_path = os.path.join(dataset_path, course_folder)
    if os.path.isdir(course_folder_path):
        for subfolder in os.listdir(course_folder_path):
            subfolder_path = os.path.join(course_folder_path, subfolder)
            if os.path.isdir(subfolder_path):
                add_orientation_elevation(subfolder_path)
print("Orientation and elevation added successfully")